# Trabajo Práctico Nro. 3 - Diseño e implementación de filtros digitales FIR e IIR

Grupo 10, 2Q2026

Integrantes:

- Ana Negre
- Juan José Medina

In [ ]:
pip install numpy matplotlib wget scipy control pandas schemdraw --quiet

In [ ]:
import os, ssl, warnings, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import signal
from scipy.io import loadmat

plt.rcParams['figure.dpi'] = 100

# display en texto plano (en el entorno del grupo el display de IPython no renderizaba)
def display(obj):
    if hasattr(obj, 'to_string'):
        print(obj.to_string(), end='\n\n')
    else:
        print(obj)

## Configuración y obtención de los datos

En primer lugar, se obtienen los archivos correspondientes al ECG y PCG utilizados en el anterior trabajo, de la misma forma que fue detallado en el mismo.

[Fuente: Pshysio.net](https://physionet.org/content/ephnogram/1.0.0/MAT/#files-panel)

In [ ]:
import os

URL = "https://physionet.org/files/ephnogram/1.0.0/MAT/"
FILE = "ECGPCG0010.mat"  # from 00 to 69

target_dir = "./data/"
os.makedirs(target_dir, exist_ok=True)

filepath = os.path.join(target_dir, FILE)

if not os.path.exists(filepath):
  !wget --no-check-certificate -P {target_dir} {URL}{FILE}

In [ ]:
from scipy.io import loadmat

data = loadmat(os.path.join(target_dir, FILE))
print(data.keys())
ecg = data['ECG'].flatten()
pcg = data['PCG'].flatten()
fs = int(data['fs'].flatten()[0])

print(f"ECG length: {len(ecg)}, sampling rate: {fs} Hz")
print(f"ECG duration: {len(ecg)/fs:.1f} seconds")
print(f"ECG samples: {len(ecg):.1f}")

print(f"PCG length: {len(pcg)}, sampling rate: {fs} Hz")
print(f"PCG duration: {len(pcg)/fs:.1f} seconds")
print(f"PCG samples: {len(pcg):.1f}")

### Preprocesamiento de las señales
Para filtrar las señales según los filtros IIR y FIR se utilizan los hallazgos y conclusiones del trabajo práctico anterior.

En el caso del PCG y ECG, las mismas están a $f_s = 8\,\mathrm{kHz}$, pero su información útil está por debajo de 150 Hz (ECG) y 400 Hz (PCG). Luego como diseñar un pasa-alto de 0.5 Hz a 8 kHz exigiría demasiados coeficientes, se **submuestrea** con filtro antialiasing (con un $M$ que mantenga holgura sobre las bandas de interés) y los filtros se diseñan a la nueva frecuencia de muestreo. En particular, se utiliza $M=16$ para el ECG y $M=4$ para el PCG.

Adicionalmente, para evaluar qué tan bien se elimina la interferencia de las señales, se "contamina" las mismas utilizando el sumador del trabajo práctico anterior. Para el ECG se utiliza la interferencia de red de 50 Hz (amplitud $0.15\,A$) en forma de senoidal y la deriva de línea base de 0.3 Hz (componentes de frecuencias bajas provenientes del movimiento/respiración del paciente, con amplitud $0.5\,A$). Para el PCG se utiliza el ruido blanco gaussiano ($\sigma = 0.05\,A$) y una interferencia de baja frecuencia de 10 Hz (amplitud $0.3\,A$) que justifica el pasa-alto.

Como $A$ se usa una amplitud de referencia de cada señal. En el ECG es el máximo ($A=0.80$, cerca de sus picos R típicos). En el PCG **no** se usa el máximo: en los 30 minutos del registro hay un artefacto aislado en $t\approx 1097.6$ s que vale $1.02$, cuatro veces los picos típicos de S1 (~0.25). Si se usara el máximo, el ruido y la interferencia de 10 Hz quedarían 12 dB más fuertes de lo previsto: el ruido tendría más potencia que todo el PCG y no se podría evaluar ningún filtro. Por eso se usa el percentil 99.9 de $|x|$ ($A=0.254$), que corresponde a los picos típicos de S1 y coincide con lo usado en el TP2 (máximo de la ventana de 2 a 6 s, $0.24$).

In [ ]:
def submuestreo(x, M):
    # decimate aplica un FIR antialiasing de fase cero antes de descartar muestras
    return signal.decimate(x, M, ftype='fir', zero_phase=True)

M_ecg, M_pcg = 16, 4
fs_e, fs_p = fs // M_ecg, fs // M_pcg            # 500 Hz y 2000 Hz

ecg_d = submuestreo(ecg, M_ecg); ecg_d = ecg_d - ecg_d.mean()
pcg_d = submuestreo(pcg, M_pcg); pcg_d = pcg_d - pcg_d.mean()
t_e = np.arange(len(ecg_d)) / fs_e
t_p = np.arange(len(pcg_d)) / fs_p

def senoidal(N, fs, f0, amp=1.0, fase=0.0):
    n = np.arange(N)
    return amp * np.cos(2 * np.pi * f0 * n / fs + fase)

def ruido_blanco(N, sigma=1.0, semilla=0):
    return np.random.default_rng(semilla).normal(0, sigma, N)

def sumador(*senales):
    return np.sum([np.asarray(x, dtype=float) for x in senales], axis=0)

# contaminación del ECG
A_e = np.max(np.abs(ecg_d))
red_50  = senoidal(len(ecg_d), fs_e, 50,  amp=0.15 * A_e)
deriva  = senoidal(len(ecg_d), fs_e, 0.3, amp=0.5 * A_e, fase=-np.pi / 2)   # arranca en 0
ecg_cont = sumador(ecg_d, red_50, deriva)


A_p = np.percentile(np.abs(pcg_d), 99.9)
blanco = ruido_blanco(len(pcg_d), sigma=0.05 * A_p)
baja   = senoidal(len(pcg_d), fs_p, 10, amp=0.3 * A_p)
pcg_cont = sumador(pcg_d, blanco, baja)

dur = len(ecg_d) / fs_e
ZOOM = (dur / 2 - 2, dur / 2 + 2)

print(f"A_ECG = {A_e:.3f} (máximo) | A_PCG = {A_p:.3f} (percentil 99.9; el máximo es {np.max(np.abs(pcg_d)):.3f} en t = {np.argmax(np.abs(pcg_d)) / fs_p:.1f} s)")
print(f"ECG: fs = {fs_e} Hz, {len(ecg_d)} muestras | PCG: fs = {fs_p} Hz, {len(pcg_d)} muestras | zoom = {ZOOM[0]:.1f}-{ZOOM[1]:.1f} s")

In [ ]:
COLORES = ['C0', 'darkorange', 'C2', 'C3', 'C4']
YLIM_PCG = (-0.5, 0.5)     # el PCG es de baja amplitud: con la escala automática queda comprimido

def graficar_tiempo(t, curvas, titulo, zoom=None, ref=None, ylim=None):
    m = slice(None) if zoom is None else (t >= zoom[0]) & (t <= zoom[1])
    fig, axs = plt.subplots(len(curvas), 1, figsize=(12, 2.2 * len(curvas)), sharex=True, sharey=True)
    axs = np.atleast_1d(axs)
    for ax, (label, x, c) in zip(axs, curvas):
        if ref is not None:
            ax.plot(t[m], ref[m], color='0.6', lw=0.8, alpha=0.8, label='original (sin contaminar)')
        ax.plot(t[m], x[m], color=c, lw=0.9, label=label)
        ax.grid(True, alpha=0.3); ax.legend(loc='upper right', fontsize=8)
        ax.tick_params(labelbottom=True)
    axs[0].set_title(titulo); axs[-1].set_xlabel('t (s)')
    if zoom is not None:
        axs[-1].set_xlim(*zoom)
    if ylim is not None:
        axs[0].set_ylim(*ylim)                     # sharey=True: vale para todos los paneles
    plt.tight_layout()
    return fig, axs

def envolvente(f, mag, fmin, fmax, log, nb=1500):
    # El registro de 30 min da cientos de miles de bins: dibujados tal cual se ven como un área rellena
    # y la última curva tapa a las demás. Se dibuja el máximo de cada intervalo (conserva los picos).
    bordes = np.geomspace(fmin, fmax, nb + 1) if log else np.linspace(fmin, fmax, nb + 1)
    idx = np.searchsorted(bordes, f) - 1
    ok = (idx >= 0) & (idx < nb)
    env = np.full(nb, -np.inf); np.maximum.at(env, idx[ok], mag[ok])
    fc = np.sqrt(bordes[:-1] * bordes[1:]) if log else (bordes[:-1] + bordes[1:]) / 2
    v = np.isfinite(env)
    return fc[v], env[v]

def graficar_espectro(fs, curvas, titulo, fmax, fmin=0, log=False):
    # módulo de la DFT en dB
    fig, ax = plt.subplots(figsize=(12, 4.5))
    tope = -np.inf
    if log:
        fmin = max(fmin, 0.1)
    for k, (label, x, c) in enumerate(curvas):
        w = np.hanning(len(x))
        X = np.fft.rfft(np.asarray(x, dtype=float) * w)
        f = np.fft.rfftfreq(len(x), 1 / fs)
        mag = 20 * np.log10(np.abs(X) / np.sum(w) + 1e-12)
        tope = max(tope, mag[(f >= max(fmin, 0.1)) & (f <= fmax)].max())
        fe, me = envolvente(f, mag, max(fmin, f[1]), fmax, log)
        ax.plot(fe, me, color=c, lw=1.6 if k == 0 else 1.1, alpha=0.9, label=label)
    ax.set_ylim(tope - 100, tope + 5)                      # 100 dB de rango, sin el piso numérico
    if log:
        ax.set_xscale('log')
    ax.set_xlim(fmin, fmax)
    ax.set_title(titulo); ax.set_xlabel('Frecuencia [Hz]'); ax.set_ylabel('|X(f)| [dB]')
    ax.grid(True, alpha=0.3, which='both'); ax.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    return fig, ax

def potencia_banda(x, fs, f1, f2):
    f, P = signal.welch(x, fs, nperseg=min(len(x), 8 * fs))
    m = (f >= f1) & (f <= f2)
    return np.sum(P[m]) * (f[1] - f[0])

def tabla_metricas(ref, cont, salidas, fs, bandas, recorte_s=2):
    r = int(recorte_s * fs); sl = slice(r, len(ref) - r)
    P0 = {b: potencia_banda(cont[sl], fs, *lim) for b, lim in bandas.items()}
    filas = {}
    for nombre, y in [('contaminada', cont)] + list(salidas.items()):
        fila = {f'ΔP {b} [dB]': 10 * np.log10(potencia_banda(y[sl], fs, *lim) / P0[b]) for b, lim in bandas.items()}
        fila['SNR vs original [dB]'] = 10 * np.log10(np.var(ref[sl]) / np.var(y[sl] - ref[sl]))
        fila['correlación'] = np.corrcoef(ref[sl], y[sl])[0, 1]
        filas[nombre] = fila
    return pd.DataFrame(filas).T.round(2)

def tabla_lineal(componentes, filtros, fs, recorte_s):
    r = int(recorte_s * fs); sl = slice(r, -r)
    filas = {}
    for nombre, fn in filtros.items():
        fila = {}
        for comp, x in componentes.items():
            y = fn(x)
            if comp.startswith('señal'):
                fila[f'{comp}: SNR de distorsión [dB]'] = 10 * np.log10(np.var(x[sl]) / np.var(y[sl] - x[sl]))
            else:
                fila[f'{comp}: atenuación [dB]'] = max(10 * np.log10(np.var(y[sl]) / np.var(x[sl])), -150)
        filas[nombre] = fila
    return pd.DataFrame(filas).T.round(1)


def filtros_de(dis, nombres=None):
    out = {}
    for n, d in dis.items():
        if nombres is not None and n not in nombres:
            continue
        out[f'{n} | FIR'] = lambda x, d=d: signal.oaconvolve(x, d['b'], mode='same')
        out[f'{n} | IIR causal'] = lambda x, d=d: filtrar_iir(d['ba'], x)
    return out

graficar_tiempo(t_e, [('ECG original', ecg_d, 'firebrick'), ('ECG contaminado (50 Hz + deriva)', ecg_cont, 'C0')],
                'ECG: señal original vs contaminada', zoom=(4, 8))
graficar_tiempo(t_p, [('PCG original', pcg_d, 'limegreen'), ('PCG contaminado (ruido blanco + 10 Hz)', pcg_cont, 'C0')],
                'PCG: señal original vs contaminada', zoom=(4, 8), ylim=YLIM_PCG)
plt.show()

In [ ]:
graficar_espectro(fs_e, [('ECG contaminado', ecg_cont, 'C0'), ('ECG original', ecg_d, 'firebrick')],
               'ECG: espectro antes de filtrar', fmax=fs_e / 2, fmin=0.1, log=True)
graficar_espectro(fs_p, [('PCG contaminado', pcg_cont, 'C0'), ('PCG original', pcg_d, 'limegreen')],
               'PCG: espectro antes de filtrar', fmax=fs_p / 2, fmin=1, log=True)
plt.show()

Vemos que en el ECG contaminado aparecen dos líneas que no están en el original: la **deriva de 0.3 Hz** (amplitud $0.5A$, la componente más grande por debajo de 1 Hz) y la **red de 50 Hz** ($0.15A$). Al ser senoidales puras, cada una aparece como un pico angosto (el ancho es el del lóbulo principal de la ventana de Hann). Fuera de esas líneas el espectro contaminado coincide con el original: como se vio en el TP2, el sumador no crea frecuencias nuevas. Esto ya anticipa las dificultades del diseño:

- La deriva está a menos de 1 Hz de la frecuencia cardíaca fundamental (~1 Hz, el pico más alto del espectro del ECG). Para separarlas hace falta una transición de unas décimas de Hz, y en un FIR eso es caro, porque $L_h\propto f_s/\Delta f$.
- La red cae dentro de la banda del QRS (8–50 Hz). Hay dos estrategias: **recortar** la banda por encima de ~40 Hz (pasa-bajo o pasa-banda) o **quitar solo los 50 Hz** (notch).

En el PCG, el ruido blanco agrega un **piso plano** en todas las frecuencias. Por debajo de **~130 Hz** queda por debajo de la señal (unos 17 dB en 50 Hz); entre ~130 y 400 Hz tienen niveles parecidos, y por encima de 400 Hz, donde el PCG casi no tiene contenido, el ruido domina. La **línea de 10 Hz** queda por debajo de la banda de máxima energía de S1 (25–45 Hz), pero en medio de la energía grave del PCG (ver la fracción de potencia por debajo de 25 Hz más abajo). También se ve un pico angosto en ~120.7 Hz que ya está en el PCG original: es una interferencia propia del registro, no de las agregadas.

Dada las características de las señales contaminadas, resulta difícil observar componentes puntuales y su atenuación. Luego se propone utilizar la potencia por banda de las señales, dado que (por Parseval) sabemos que la energía en el tiempo equivale a la suma de la energía en cada banda de frecuencia.

Con esto puede verificarse cuánta potencia de la señal se conserva en cada banda luego de su procesamiento. De esta forma, se podrá medir cuánto filtra cada filtro sin juzgarlo "a ojo". Además permite comparar la fraccion según frecuencia entre las señales originales y contaminadas:

In [ ]:
# Fracción de la potencia de cada señal original por encima/debajo de los cortes
def fraccion(x, fs, f1, f2):
    return 100 * potencia_banda(x, fs, f1, f2) / potencia_banda(x, fs, 0, fs / 2)
print(f"ECG original: {fraccion(ecg_d, fs_e, 0, 0.5):.2f} % de la potencia < 0.5 Hz | "
      f"{fraccion(ecg_d, fs_e, 40, fs_e/2):.2f} % > 40 Hz | {fraccion(ecg_d, fs_e, 150, fs_e/2):.3f} % > 150 Hz")
print(f"PCG original: {fraccion(pcg_d, fs_p, 0, 25):.2f} % de la potencia < 25 Hz | "
      f"{fraccion(pcg_d, fs_p, 25, 45):.2f} % entre 25 y 45 Hz | {fraccion(pcg_d, fs_p, 400, fs_p/2):.3f} % > 400 Hz")

## Diseño de filtros
Para procesar las señales se utilizarán dos familias de filtros: los **FIR** (finite impulse response) e **IIR** (infinite impulse response).

### FIR
En primer lugar, se tienen los filtros FIR. Los mismos tendrán una función de transferencia que puede expresarse como:
$$H(z)=\sum_{k=0}^{N-1} b[k]\,z^{-k}$$
Vemos que, al solo tener polos en el origen, la ROC de la transformada siempre contendrá la circunferencia unidad, y por ende estos filtros siempre serán estables.

En particular, para implementar estos filtros se utilizará el método del ventaneo. Se parte del filtro ideal pasabajo, donde:
$$\hat{H}(f)=\Pi(\frac{f}{2 f_c})$$
$$h_c(t)=2 f_c sinc(2 f_c t)$$
Este filtro analógico no es realizable, dado que $h_c$ es de duración infinita y no causal. Luego se muestrea $h_c$ con período T ($h[n]=T\,h_c(nT)=\frac{\Omega_c}{\pi}\mathrm{sinc}\big(\frac{\Omega_c}{\pi}n\big)$; el factor $T$ compensa el $1/T$ del muestreo), se trunca a $L_h$ muestras y se lo desplaza $D=\frac{L_h - 1}{2}$. Para el truncado, se multiplica la respuesta al impulso por una ventana $w[n]$, lo cual cambiará las características del filtro según la ventana utilizada (rectangular, Hamming, Hann, Blackman, Kaiser). El lóbulo principal de la ventana fija el ancho de la transición y los lóbulos laterales fijan el ripple y la atenuación en la banda de rechazo. Cada ventana es un compromiso entre ambos, y para una ventana dada la transición se angosta al aumentar $L_h$: $\Delta\Omega = C\pi/(L_h-1)$. Aumentar $L_h$ no reduce el ripple (fenómeno de Gibbs): para eso hay que cambiar de ventana. La cantidad de muestras tiene además un efecto directo sobre la _fase_: como $h[n]$ es simétrica, el retardo de grupo es constante y vale $\tau_g = \frac{L_h-1}{2}$ muestras.
El orden del filtro es $N= L_h - 1$.

### IIR
Por el otro lado, en el caso de los filtros IIR, la función de transferencia es un cociente de polinomios en $z^{-1}$:
$$H(z)=\frac{\sum_{k=0}^{M} b[k]\,z^{-k}}{1+\sum_{k=1}^{N} a[k]\,z^{-k}}=\frac{B(z)}{A(z)}\qquad\Longleftrightarrow\qquad y[n]=\sum_{k=0}^{M} b[k]\,x[n-k]-\sum_{k=1}^{N} a[k]\,y[n-k]$$

A diferencia de los FIR, tienen polos fuera del origen: la salida depende de entradas *y* de salidas pasadas, y la respuesta al impulso es de duración infinita. Luego a diferencia del FIR, la estabilidad no está garantizada por la estructura: el sistema solo es estable si todos sus polos están dentro del círculo unidad. Como se ve más abajo, la transformación bilineal la garantiza en teoría, pero igual hay que verificarla en cada diseño, porque el redondeo de los coeficientes puede mover polos que están muy cerca de $|z|=1$.

Luego para implementar estos filtros, se parte de un filtro analógico $H_a(s)=B(s)/A(s)$ y se convierte a uno digital $H(z)$. Sabemos que un sistema analógico causal es estable si y solo si todos sus polos cumplen $\mathrm{Re}\{s\}<0$. Se busca entonces una transformación $s\leftrightarrow z$ tal que:

1. el eje $i\omega$ del plano $s$ se corresponda con la circunferencia unidad del plano $z$ (relación directa entre las respuestas en frecuencia de ambos dominios);
2. el semiplano izquierdo del plano $s$ se mapee en el interior de la circunferencia unidad (un filtro estable se transforma en un filtro estable).

Para lograr esto se aproxima la integral con la regla del trapecio (transformación bilineal): si $w[n]$ aproxima a $y'(nT)$, entonces:
$$y[n]-y[n-1]=\frac{T}{2}\,\bigl(w[n-1]+w[n]\bigr)\quad\Rightarrow\quad H(z)=\frac{W(z)}{Y(z)}=\frac{2}{T}\,\frac{1-z^{-1}}{1+z^{-1}}$$

Como el derivador continuo es $H(s)=s$, resulta
$$s=\frac{2}{T}\,\frac{1-z^{-1}}{1+z^{-1}}=\frac{2}{T}\,\frac{z-1}{z+1}\qquad\Longleftrightarrow\qquad z=\frac{1+\frac{T}{2}s}{1-\frac{T}{2}s}\qquad H(z)=H_a(s)\Big|_{s=\frac{2}{T}\frac{z-1}{z+1}}$$

Con $s=\sigma+i\omega$ y $z=re^{i\Omega}$:
$$\sigma=\frac{2}{T}\,\frac{r^2-1}{r^2+2r\cos\Omega+1}\qquad\Rightarrow\qquad r<1\Leftrightarrow\sigma<0,\quad r=1\Leftrightarrow\sigma=0,\quad r>1\Leftrightarrow\sigma>0$$
ya que el denominador es $|z+1|^2>0$. Por lo tanto, el semiplano izquierdo se mapea en el interior de la circunferencia unidad (**estable $\Rightarrow$ estable**) y el eje imaginario en la circunferencia unidad.

Respecto a la relación entre frecuencias, vemos que sobre la circunferencia unidad, $z=e^{i\Omega}$ y $s=i\omega$:
$$i\omega=\frac{2}{T}\,\frac{e^{i\Omega}-1}{e^{i\Omega}+1}=i\,\frac{2}{T}\tan\!\left(\frac{\Omega}{2}\right)\qquad\Longleftrightarrow\qquad \omega=\frac{2}{T}\tan\!\left(\frac{\Omega}{2}\right),\quad \Omega=2\arctan\!\left(\frac{\omega T}{2}\right)$$
Luego todo el eje $0\le\omega<\infty$ se comprime en $0\le\Omega<\pi$, por lo que **no hay aliasing**, pero la relación es no lineal (se deforma el eje de frecuencias, *warping*): solo para $\omega T\ll1$ vale $\Omega\approx\omega T$, y a frecuencias altas la respuesta se distorsiona sobre el eje de frecuencias.
Se conserva la forma por tramos de $|H_a|$ (las bandas planas siguen planas) pero se desplazan sus bordes. Por eso la bilineal es adecuada para filtros con respuesta constante por tramos (pasa-bajo, pasa-alto, pasa-banda, rechaza-banda), pero **no conserva la fase lineal** ni la forma de la respuesta al impulso.

Para que una frecuencia crítica (por ejemplo, la de corte) quede en su lugar luego de la transformación, el prototipo analógico se diseña en la frecuencia **predistorsionada** $\omega_a=\frac{2}{T}\tan\!\left(\frac{\Omega_c}{2}\right)$, con $\Omega_c=2\pi f_cT$. Así la frecuencia crítica cae exactamente en su lugar después de la bilineal. En este notebook la función `predistorsion()` aplica esto a los bordes $f_p$ y $f_{st}$ antes de diseñar el prototipo analógico, y después `signal.bilinear` hace $s=\frac{2}{T}\frac{z-1}{z+1}$.

### Filtros utilizados

Se aplican las bandas de corte provenientes del análisis del trabajo práctico anterior, provenientes de las distintas ondas relevantes de las señales (P, QRS, T, etc.).
Siguiendo el procedimiento de la teoría, cada filtro se especifica según los requerimientos: borde de la banda de paso $f_p$, borde de la banda de rechazo $f_{st}$, ripple máximo $r_p$ y atenuación mínima $\alpha_s$. De ahí salen la frecuencia de corte del prototipo ($f_c=\frac{f_p+f_{st}}{2}$), el largo $L_h$ del FIR y el orden $N$ del IIR.


| Señal | Filtro | $f_p$ [Hz] | $f_{st}$ [Hz] | $f_c$ [Hz] | Justificación (TP2)                                                                                                                       |
|---|---|---|---|---|-------------------------------------------------------------------------------------------------------------------------------------------|
| ECG | Pasa-bajo | 30 | 50 | 40 | Conserva P (5–30 Hz), T (0–10 Hz) y la mayor parte del QRS (centrado en 10 Hz). $f_{st}=50$ Hz deja la red justo en la banda de rechazo.  |
| ECG | Pasa-alto | 0.7 | 0.3 | 0.5 | La deriva (0.3 Hz) queda en la banda de rechazo; 0.7 Hz ≈ 42 lpm, por debajo de la frecuencia cardíaca, así que el ritmo pasa             |
| ECG | Pasa-banda | 0.7–30 | 0.3 / 50 | 0.5–40 | Combina los dos anteriores: saca la deriva y la red                                                                                       |
| ECG | Pasa-banda QRS | 8–12 | 2 / 18 | 5–15 | Banda del QRS (Pan y Tompkins, 1985), centrada en 10 Hz; útil para detectar picos R                                                       |
| ECG | Rechaza-banda (notch) | ≤47 y ≥53 | 49–51 | 48–52 | Saca la red (con ±1 Hz de tolerancia) sin recortar el ancho de banda del ECG                                                              |
| PCG | Pasa-bajo | 350 | 450 | 400 | Los soplos llegan hasta ~400 Hz; por encima domina el ruido                                                                               |
| PCG | Pasa-alto | 35 | 15 | 25 | La interferencia de 10 Hz queda en la banda de rechazo. Compromiso con S1 (máximo entre 25 y 45 Hz), que se cuantifica en la sección del PCG            |
| PCG | Pasa-banda | 35–350 | 15 / 450 | 25–400 | Banda útil del PCG (S1, S2 y soplos)                                                                                                      |


Para los FIR se usa siempre $h[n]$ simétrica con $L_h$ impar, es decir orden $N=L_h-1$ par (tipo I): con $L_h$ par (tipo II) hay un cero en $z=-1$ y no se pueden hacer pasa-altos ni rechaza-banda. Los nombres de los filtros conservan la $f_c$ del prototipo. Para el FIR, $f_c$ es el punto de −6 dB; para el IIR, la plantilla fija $f_p$ y $f_{st}$, y su punto de −3 dB queda entre ambos.

Los parámetros comunes para ripple y atenuación mínima en la banda de rechazo, haciendo uso de la tabla provista en clase para el diseño de filtros FIR:
$$r_p=0.1\ \mathrm{dB}\qquad \alpha_s= 50\ \mathrm{dB}$$
Con estos valores, según la tabla de la teoría, la ventana de menor lóbulo principal que cumple es **Hamming**: Hann llega solo a 43.9 dB.

In [ ]:
RP, AS = 0.1, 50        # ripple máximo en la banda de paso y atenuación mínima en la de rechazo [dB]

# fp: borde(s) de la banda de paso, fst: borde(s) de la banda de rechazo [Hz]
espec_ecg = {
    'Pasa-bajo 40 Hz':        dict(btype='lowpass',  fp=30,        fst=50,        xlim=(0, 100),    log=False),
    'Pasa-alto 0.5 Hz':       dict(btype='highpass', fp=0.7,       fst=0.3,       xlim=(0.05, 20),  log=True),
    'Pasa-banda 0.5-40 Hz':   dict(btype='bandpass', fp=[0.7, 30], fst=[0.3, 50], xlim=(0.05, 250), log=True),
    'Pasa-banda QRS 5-15 Hz': dict(btype='bandpass', fp=[8, 12],   fst=[2, 18],   xlim=(0, 60),     log=False),
    'Notch 50 Hz':            dict(btype='bandstop', fp=[47, 53],  fst=[49, 51],  xlim=(30, 70),    log=False),
}

espec_pcg = {
    'Pasa-bajo 400 Hz':       dict(btype='lowpass',  fp=350,        fst=450,        xlim=(0, 1000), log=False),
    'Pasa-alto 25 Hz':        dict(btype='highpass', fp=35,         fst=15,         xlim=(1, 200),  log=True),
    'Pasa-banda 25-400 Hz':   dict(btype='bandpass', fp=[35, 350],  fst=[15, 450],  xlim=(1, 1000), log=True),
}

# Tabla de la teoría (método de ventanas)!! Muestras según ancho de transición, ripple, etc según tipo de ventana, etc.
TABLA_VENTANAS = pd.DataFrame(
    {'lóbulo principal': ['4π/Lh', '8π/Lh', '8π/Lh', '12π/Lh'],
     'lateral máx. [dB]': [-13.3, -31.5, -42.7, -58.1],
     'C': [1.84, 6.22, 6.64, 11.13],
     'αs [dB]': [20.9, 43.9, 54.5, 75.3],
     'rp [dB]': [1.57, 0.11, 0.033, 0.0030]},
    index=['boxcar', 'hann', 'hamming', 'blackman'])

def bandas(e):
    fp, fst = np.atleast_1d(e['fp']).astype(float), np.atleast_1d(e['fst']).astype(float)
    fc = (fp + fst) / 2                          # paso 2: Ωc = (Ωp + Ωs) / 2  (centro de la transición)
    Df = np.min(np.abs(fp - fst))                # pasa/rechaza-banda: se diseña con la transición más angosta
    return (fc[0] if fc.size == 1 else fc), Df

def elegir_ventana(As=AS, Rp=RP):
    # paso 3: la de lóbulo principal más angosto (menor C) que cumple αs y rp
    ok = TABLA_VENTANAS[(TABLA_VENTANAS['αs [dB]'] >= As) & (TABLA_VENTANAS['rp [dB]'] <= Rp)]
    return ok['C'].idxmin()

def largo_fir(Df, fs, C):
    # paso 4: Lh >= 1 + Cπ/ΔΩ con ΔΩ = 2π Δf / fs  =>  Lh >= 1 + C fs / (2 Δf); impar => tipo I
    Lh = int(np.ceil(1 + C * fs / (2 * Df)))
    return Lh + (1 - Lh % 2)

print(f"Plantilla: rp <= {RP} dB, αs >= {AS} dB  ->  ventana elegida: {elegir_ventana()}")
display(TABLA_VENTANAS)

A continuación se diseñan los filtros especificados, utilizando las frecuencias de muestreo de las señales submuestreadas, $f_s^{ECG}=500$ Hz y $f_s^{PCG}=2000$ Hz. Las especificaciones obtenidas se comparan con los parámetros teóricos.

In [ ]:
def freqz_ba(ba, worN, fs):
    # H(z) = Π B_i(z)/A_i(z)
    H = 1
    for b, a in ba:
        f, Hi = signal.freqz(b, a, worN=worN, fs=fs)
        H = H * Hi
    return f, H

def filtrar_iir(ba, x):
    for b, a in ba:
        x = signal.lfilter(b, a, x)
    return x

def polos(ba):
    return np.concatenate([np.roots(a) for _, a in ba])

def ceros(ba):
    return np.concatenate([np.roots(b) for b, _ in ba])

def estable(ba, n=2**17):
    x = np.zeros(n); x[0] = 1
    with np.errstate(all='ignore'):
        h = np.abs(filtrar_iir(ba, x))
    return bool(np.all(np.isfinite(h)) and h[-n // 8:].max() < 1e-3 * h.max())

def cumple(ba, e, fs, Rp=RP, As=AS):
    rp, a_s = medir_plantilla(e, fs, ba=ba)
    return bool(rp <= Rp + 0.01 and a_s >= As - 0.5)

def mult_iir(ba):
    # forma directa: N+1 coeficientes b_k y N coeficientes a_k (a_0 = 1) por etapa
    return sum(len(b) + len(a) - 1 for b, a in ba)

def medir_plantilla(e, fs, b=None, ba=None, n=2**16):
    # ripple en la banda de paso y atenuación mínima en la de rechazo, medidos sobre |H|
    f, H = signal.freqz(b, 1, worN=n, fs=fs) if ba is None else freqz_ba(ba, n, fs)
    m = 20 * np.log10(np.abs(H) + 1e-15)
    paso, rech = bandas_paso_rechazo(f, e)
    return m[paso].max() - m[paso].min(), -m[rech].max()

def bandas_paso_rechazo(f, e):
    fp, fst, bt = np.atleast_1d(e['fp']), np.atleast_1d(e['fst']), e['btype']
    if bt == 'lowpass':
        return f <= fp[0], f >= fst[0]
    if bt == 'highpass':
        return f >= fp[0], f <= fst[0]
    if bt == 'bandpass':
        return (f >= fp[0]) & (f <= fp[1]), (f <= fst[0]) | (f >= fst[1])
    return (f <= fp[0]) | (f >= fp[1]), (f >= fst[0]) & (f <= fst[1])

def disenar_fir(e, fs, Rp=RP, As=AS):
    fc, Df = bandas(e)
    ventana = elegir_ventana(As, Rp)
    Lh_teo = Lh = largo_fir(Df, fs, TABLA_VENTANAS.loc[ventana, 'C'])
    while True:
        # firwin calcula h[n] = sinc ideal retardada (Lh-1)/2 muestras × ventana (ver verificación más abajo)
        b = signal.firwin(Lh, fc, window=ventana, pass_zero=e['btype'], fs=fs)
        rp, a_s = medir_plantilla(e, fs, b=b)
        if (rp <= Rp and a_s >= As) or Lh > 5 * Lh_teo:
            break
        Lh += 2                                  # paso 6: si no cumple, se alarga (manteniéndolo impar)
    return dict(b=b, fc=fc, ventana=ventana, Lh_teo=Lh_teo, rp_fir=rp, As_fir=a_s)

def predistorsion(f, fs):
    return 2 * fs * np.tan(np.pi * np.asarray(f, dtype=float) / fs)

def bilineal(fp, fst, btype, fs, Rp, As, familia):
    wp, ws = predistorsion(fp, fs), predistorsion(fst, fs)
    if familia == 'butter':
        N, Wn = signal.buttord(wp, ws, Rp, As, analog=True)
        bs, as_ = signal.butter(N, Wn, btype, analog=True)
    elif familia == 'cheby1':
        N, Wn = signal.cheb1ord(wp, ws, Rp, As, analog=True)
        bs, as_ = signal.cheby1(N, Rp, Wn, btype, analog=True)
    else:
        N, Wn = signal.ellipord(wp, ws, Rp, As, analog=True)
        bs, as_ = signal.ellip(N, Rp, As, Wn, btype, analog=True)
    b, a = signal.bilinear(bs, as_, fs)
    return (b, a), N * (2 if btype in ('bandpass', 'bandstop') else 1)

def disenar_iir(e, fs, Rp=RP, As=AS, familia='ellip'):
    bt = e['btype']
    ba, orden = bilineal(e['fp'], e['fst'], bt, fs, Rp, As, familia)
    ba = [ba]
    if bt == 'bandpass':
        # alternativa: pasa-alto seguido de pasa-bajo, cada uno con su propio par (b, a) y la mitad del ripple.
        # Se usa si es más barata o si la forma directa B(z)/A(z) deja de ser estable o de cumplir la plantilla por redondeo.
        hp, Nh = bilineal(e['fp'][0], e['fst'][0], 'highpass', fs, Rp / 2, As, familia)
        lp, Nl = bilineal(e['fp'][1], e['fst'][1], 'lowpass', fs, Rp / 2, As, familia)
        directa_ok = estable(ba) and cumple(ba, e, fs, Rp, As)
        if Nh + Nl < orden or (not directa_ok and estable([hp, lp])):
            ba, orden = [hp, lp], Nh + Nl
    return ba, orden

# ---------- polos "exactos": el mismo diseño hecho con polos y ceros (zpk), sin pasar por los polinomios
def polos_exactos(e, fs, Rp=RP, As=AS, familia='butter'):
    wp, ws = predistorsion(e['fp'], fs), predistorsion(e['fst'], fs)
    if familia == 'butter':
        N, Wn = signal.buttord(wp, ws, Rp, As, analog=True)
        z, p, k = signal.butter(N, Wn, e['btype'], analog=True, output='zpk')
    else:
        N, Wn = signal.ellipord(wp, ws, Rp, As, analog=True)
        z, p, k = signal.ellip(N, Rp, As, Wn, e['btype'], analog=True, output='zpk')
    return signal.bilinear_zpk(z, p, k, fs)[1]

def disenar_todos(especificaciones, fs, familia='ellip'):
    dis = {}
    for nombre, e in especificaciones.items():
        ba, N = disenar_iir(e, fs, familia=familia)
        dis[nombre] = dict(e, **disenar_fir(e, fs), ba=ba, N=N)
    return dis

In [ ]:
# Primer intento: IIR Butterworth en forma directa H(z) = B(z)/A(z)
x_prueba = ruido_blanco(60 * 2000)          # ruido de prueba, σ = 1

def prueba_ba(espec, fs, familia):
    filas = {}
    for nombre, e in espec.items():
        (b, a), N = bilineal(e['fp'], e['fst'], e['btype'], fs, RP, AS, familia)     # forma directa, sin cascada
        p_ba, p_ex = np.abs(np.roots(a)), np.abs(polos_exactos(e, fs, familia=familia))
        with np.errstate(all='ignore'):
            y = signal.lfilter(b, a, x_prueba[:int(60 * fs)])
        ymax, est = np.max(np.abs(y)), estable([(b, a)])
        rp, a_s = medir_plantilla(e, fs, ba=[(b, a)]) if est else (np.nan, np.nan)
        filas[nombre] = {'orden': N, 'max|a_k|': f"{np.max(np.abs(a)):.1e}",
                         '|p| máx exacto (zpk)': round(p_ex.max(), 6),
                         '|p| máx np.roots(a)': round(p_ba.max(), 6),
                         'h[n] decae (estable)': est,
                         'max|y| ruido σ=1, 60 s': 'inf / nan' if not np.isfinite(ymax) else f"{ymax:.2e}",
                         'rp [dB]': round(rp, 3), 'αs [dB]': round(a_s, 1),
                         'cumple plantilla': est and cumple([(b, a)], e, fs)}
    return pd.DataFrame(filas).T

print(f"Butterworth, rp = {RP} dB, αs = {AS} dB, forma directa B(z)/A(z)")
print(f"ECG (fs = {fs_e} Hz)"); display(prueba_ba(espec_ecg, fs_e, 'butter'))
print(f"PCG (fs = {fs_p} Hz)"); display(prueba_ba(espec_pcg, fs_p, 'butter'))

Vemos que el Butterworth necesita órdenes altos, entre 10 y 42: es la familia más plana y por eso la de transición más lenta, y su orden lo fija el cociente $f_{st}/f_p$, porque cae $20N$ dB por década. En forma directa $B(z)/A(z)$, 5 de los 8 filtros fallan: el pasa-alto de 0.5 Hz, el pasa-banda 0.5–40 Hz, el pasa-banda QRS y el pasa-banda del PCG son inestables ($h[n]$ crece sin límite), y el notch es estable pero no cumple la plantilla (prácticamente no atenúa en la banda de rechazo).

La causa **no es la familia sino la forma directa**. Con muchos polos agrupados cerca de $z=1$, las raíces de $A(z)$ son extremadamente sensibles a los coeficientes, y el redondeo de doble precisión alcanza para sacar polos del círculo unidad. Se ve en la tabla: los polos exactos (calculados en forma `zpk`, sin pasar por el polinomio) están todos dentro, pero los de `np.roots(a)` no. Hay dos soluciones posibles: implementar el mismo Butterworth como cascada de secciones de segundo orden (SOS), o usar una familia que cumpla la plantilla con orden más bajo. Se elige la segunda: la familia **elíptica**, que con los mismos requerimientos necesita órdenes de 5 a 12.

Notar que el elíptico tampoco resuelve todo en forma directa: el pasa-banda del ECG (orden 10) deja de cumplir la plantilla (ripple de varios dB en la banda de paso), por eso más abajo se implementa en dos etapas $(b,a)$. Los valores exactos de esa fila (ripple, polo según `np.roots`) cambian de una computadora a otra y entre versiones de numpy/scipy: están al límite de la precisión numérica, lo que muestra lo frágil que es la forma directa. Por la misma razón `np.roots(a)` no es un diagnóstico confiable (en algunas versiones da $|p|>1$ aunque $h[n]$ decae): la verificación confiable es la respuesta medida ($h[n]$ y la plantilla).

In [ ]:
print(f"Elíptico, misma plantilla, forma directa B(z)/A(z)")
print(f"ECG (fs = {fs_e} Hz)"); display(prueba_ba(espec_ecg, fs_e, 'ellip'))
print(f"PCG (fs = {fs_p} Hz)"); display(prueba_ba(espec_pcg, fs_p, 'ellip'))

In [ ]:
def resumen(dis, fs):
    filas = {}
    for n, d in dis.items():
        p = polos(d['ba'])
        rp_i, as_i = medir_plantilla(d, fs, ba=d['ba'])
        filas[n] = {'fp / fst [Hz]': f"{d['fp']} / {d['fst']}",
                    'fc FIR (-6 dB) [Hz]': d['fc'],
                    'FIR: Lh teoría': d['Lh_teo'], 'FIR: Lh final': len(d['b']),
                    'FIR: rp [dB]': round(d['rp_fir'], 3), 'FIR: αs [dB]': round(d['As_fir'], 1),
                    'FIR: retardo [ms]': 1000 * (len(d['b']) - 1) / 2 / fs,
                    'IIR: orden': d['N'], 'IIR: etapas (b, a)': len(d['ba']),
                    'IIR: rp [dB]': round(rp_i, 3), 'IIR: αs [dB]': round(as_i, 1),
                    'IIR: |polo| máx': round(np.max(np.abs(p)), 6),
                    'IIR estable': estable(d['ba']),
                    'mult/muestra FIR': len(d['b']), 'mult/muestra IIR': mult_iir(d['ba'])}
    return pd.DataFrame(filas).T

dis_ecg = disenar_todos(espec_ecg, fs_e)        # FIR Hamming + IIR elíptico (b, a)
dis_pcg = disenar_todos(espec_pcg, fs_p)

print(f"ECG (fs = {fs_e} Hz), ventana {dis_ecg['Pasa-bajo 40 Hz']['ventana']}, IIR elíptico"); display(resumen(dis_ecg, fs_e))
print(f"PCG (fs = {fs_p} Hz), ventana {dis_pcg['Pasa-bajo 400 Hz']['ventana']}, IIR elíptico"); display(resumen(dis_pcg, fs_p))

Finalmente vemos que todos los filtros cumplen la plantilla. Para los FIR, $r_p$ está entre $0.03$ y $0.04$ dB (la teoría da 0.033 dB para Hamming) y $\alpha_s$ entre $51$ y $53$ dB. Para los IIR, $r_p\le 0.1$ dB y $\alpha_s = 50$ dB: el elíptico cumple justo, porque reparte el ripple de forma pareja en ambas bandas. En el pasa-banda QRS el FIR necesitó alargarse de 279 a 283 coeficientes para cumplir (paso 6 de la teoría), porque en el pasa-banda se suman las colas de las dos copias del espectro.

Se observa la diferencia de costo entre los dos tipos de filtro en el pasa-alto y el pasa-banda del ECG: el FIR requiere **4151** coeficientes, con un retardo de **4.15 s**, mientras que el IIR elíptico equivalente es de orden 5 (11 multiplicaciones por muestra en forma directa) y de orden 11 en dos etapas para el pasa-banda (24 multiplicaciones). El Butterworth, en cambio, habría necesitado órdenes 10 y 28 para lo mismo (tabla anterior) y en forma directa no funciona.

Finalmente, vemos que todos los polos cumplen $|p|<1$, como garantiza la bilineal (en los FIR todos los polos están en el origen). Por ende todos estos filtros forman sistemas estables. Esto puede verse graficando los polos y ceros de estos filtros. Por ejemplo, para el pasa-banda QRS, el notch de 50 Hz y el pasa-bajo de 40 Hz:

In [ ]:
def plot_polos_ceros(ba, titulo, ax):
    z, p = ceros(ba), polos(ba)
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, ls='--', color='k', lw=0.8))
    ax.plot(z.real, z.imag, 'o', mfc='none', color='C0', ms=8, label='ceros (raíces de B)')
    ax.plot(p.real, p.imag, 'x', color='C3', ms=8, label='polos (raíces de A)')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper left', fontsize=8)
    ax.set_title(titulo, fontsize=10); ax.set_xlabel('Re'); ax.set_ylabel('Im')
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
plot_polos_ceros(dis_ecg['Pasa-banda QRS 5-15 Hz']['ba'], f"ECG: pasa-banda QRS (elíptico orden {dis_ecg['Pasa-banda QRS 5-15 Hz']['N']})", axs[0])
plot_polos_ceros(dis_ecg['Notch 50 Hz']['ba'], f"ECG: notch 48-52 Hz (elíptico orden {dis_ecg['Notch 50 Hz']['N']})", axs[1])
plot_polos_ceros(dis_ecg['Pasa-bajo 40 Hz']['ba'], f"ECG: pasa-bajo 40 Hz (elíptico orden {dis_ecg['Pasa-bajo 40 Hz']['N']})", axs[2])
plt.tight_layout()

# ángulos de los ceros sobre el círculo unidad, en Hz
for n in ['Pasa-bajo 40 Hz', 'Pasa-banda QRS 5-15 Hz', 'Notch 50 Hz']:
    z = ceros(dis_ecg[n]['ba']); p = polos(dis_ecg[n]['ba'])
    fz = np.sort(np.abs(np.angle(z[z.imag >= -1e-12]))) * fs_e / (2 * np.pi)
    print(f"{n}: ceros en f = {np.round(fz, 2)} Hz, |z| = {np.round(np.abs(z).min(), 6)}..{np.round(np.abs(z).max(), 6)}; "
          f"|polo| máx = {np.abs(p).max():.4f}")

Todos los polos están dentro del círculo unidad. A diferencia del Butterworth, cuyos ceros están todos en $z=\pm1$, los **ceros del elíptico están sobre el círculo unidad dentro de la banda de rechazo** (por ejemplo, en 51.4, 75.2 y 250 Hz para el pasa-bajo). Cada cero anula una frecuencia, y entre ceros la respuesta sube hasta −50 dB: de ahí el rechazo equiripple. En el notch, los polos están en el mismo ángulo que los ceros, apenas hacia adentro, y por eso la ganancia vuelve a 1 enseguida a ambos lados de la ranura. En el QRS los polos y ceros se amontonan cerca de $z=1$ porque $f \ll f_s$, y a esta escala no se distinguen.

En el pasa-banda QRS hay ceros en 5.81 y 16.51 Hz, **dentro** de las bandas de transición de la plantilla (2–8 y 12–18 Hz). No es un error. La transformación pasa-bajo → pasa-banda del prototipo analógico es simétrica en escala geométrica alrededor de $\sqrt{8\cdot12}\approx 9.8$ Hz, así que, para cumplir el borde más exigente, el filtro termina siendo más selectivo de lo pedido del otro lado: ya atenúa al menos 50 dB por debajo de ~6 Hz y por encima de ~16 Hz. Por eso su banda de paso real es más angosta que la del FIR (−3 dB en 7.7–12.5 Hz contra 5.7–14.3 Hz).

Luego para verificar el comportamiento de estos filtros puede graficarse su respuesta en frecuencia para verificar que cumplan lo pedido, comparando así FIR e IIR para cada tipo.
 Las referencias son: punteada negra, −3 dB; gris, $f_c$; verde, $f_p$; roja, $f_{st}$ y la línea de −50 dB de la plantilla. El retardo de grupo se grafica solo donde $|H|>-40$ dB: en la banda de rechazo no tiene sentido físico y la derivada de la fase es ruido numérico. Para el FIR se usa el valor exacto $(L_h-1)/2$, que vale por la simetría. En los gráficos con eje de frecuencias logarítmico, la fase del FIR se ve curva aunque es lineal: es una recta en $f$, no en $\log f$. La escala de la fase se ajusta al rango visible.

In [ ]:
def respuesta(coef, tipo, fs, n=2**15):
    if tipo == 'fir':
        w, h = signal.freqz(coef, 1, worN=n, fs=fs)
        gd = np.full(len(w), (len(coef) - 1) / 2)          # simétrico => τg = (Lh−1)/2 exacto (teoría)
    else:
        w, h = freqz_ba(coef, n, fs)                     # coef: lista de etapas (b, a)
        gd = -np.gradient(np.unwrap(np.angle(h)), 2 * np.pi * w / fs)   # τg = −dθ/dΩ [muestras]
    return w, h, np.unwrap(np.angle(h)), gd

def graficar_respuestas(curvas, fs, titulo, xlim=None, log=False, ylim=(-100, 5), fcs=None,
                        paneles=('mag', 'fase', 'gd'), n=2**15, plantilla=None):
    # curvas: lista de (etiqueta, 'fir'|'iir', coef)  ->  fir: vector b ; iir: lista de etapas (b, a)
    nombres = {'mag': 'Magnitud [dB]', 'fase': 'Fase [rad]', 'gd': 'Retardo de grupo [ms]'}
    fig, axs = plt.subplots(len(paneles), 1, figsize=(11, 3.2 * len(paneles)), sharex=True)
    axs = np.atleast_1d(axs)
    for label, tipo, coef in curvas:
        w, h, fase, gd = respuesta(coef, tipo, fs, n)
        mag = 20 * np.log10(np.abs(h) + 1e-12)
        datos = {'mag': mag, 'fase': fase, 'gd': np.where(mag > -40, 1000 * gd / fs, np.nan)}  # τg solo donde pasa señal
        for ax, p in zip(axs, paneles):
            ax.plot(w, datos[p], lw=1.1, label=label)
    for ax, p in zip(axs, paneles):
        ax.set_ylabel(nombres[p]); ax.grid(True, alpha=0.3, which='both')
        if p == 'mag':
            ax.set_ylim(*ylim); ax.axhline(-3, color='k', ls=':', lw=0.8)
            if plantilla is not None:
                ax.axhline(-AS, color='C3', ls=':', lw=1, label=f'requerimientos: −{AS} dB, fp (verde), fst (rojo)')
                for f_ in np.atleast_1d(plantilla['fp']):
                    ax.axvline(f_, color='C2', ls='-.', lw=0.9)
                for f_ in np.atleast_1d(plantilla['fst']):
                    ax.axvline(f_, color='C3', ls='-.', lw=0.9)
        if fcs is not None:
            for fc in np.atleast_1d(fcs):
                ax.axvline(fc, color='gray', ls='--', lw=0.8)
    axs[0].set_title(titulo); axs[0].legend(loc='lower right', fontsize=8)
    axs[-1].set_xlabel('Frecuencia [Hz]')
    if log:
        axs[-1].set_xscale('log')
    if xlim is not None:
        axs[-1].set_xlim(*xlim)
        # la fase desenvuelta del FIR largo llega a miles de rad fuera del zoom: se escala con lo visible
        for ax, p in zip(axs, paneles):
            if p != 'mag':
                vis = []
                for l in ax.get_lines():
                    xd, yd = np.asarray(l.get_xdata(), float), np.asarray(l.get_ydata(), float)
                    if len(xd) > 10:                       # solo curvas, no las líneas de referencia
                        vis.append(yd[(xd >= xlim[0]) & (xd <= xlim[1])])
                vis = np.concatenate([v[np.isfinite(v)] for v in vis if len(v)]) if vis else np.array([])
                if vis.size:
                    lo, hi = vis.min(), vis.max(); pad = 0.05 * (hi - lo + 1e-9)
                    ax.set_ylim(lo - pad, hi + pad)
    plt.tight_layout()
    return fig, axs

def comparar_fir_iir(dis, fs, señal):
    for nombre, d in dis.items():
        curvas = [(f"FIR {d['ventana'].capitalize()} (Lh = {len(d['b'])})", 'fir', d['b']),
                  (f"IIR elíptico (orden {d['N']})", 'iir', d['ba'])]
        graficar_respuestas(curvas, fs, f"{señal}: {nombre} - FIR vs IIR", xlim=d['xlim'], log=d['log'],
                            fcs=d['fc'], plantilla=d)

In [ ]:
comparar_fir_iir(dis_ecg, fs_e, 'ECG')
comparar_fir_iir(dis_pcg, fs_p, 'PCG')

Luego, sobre cada filtro se mide:

- las frecuencias de −3 y −6 dB;
- el ancho de transición real (entre −0.1 y −50 dB), que se compara con la teoría, $\Delta f = C f_s/(2(L_h-1))$;
- el retardo de grupo;
- la duración del transitorio del IIR, es decir, el tiempo hasta que $|h[n]|$ queda por debajo del 0.1 % de su máximo.

Con estos valores se puede saber cuánto se recorta en cada extremo mediante las métricas de potencia en las bandas que se verán más adelante.

In [ ]:
def cruce(f, mag, nivel, desde, hasta):
    # primera frecuencia, yendo de 'desde' (lado de paso) hacia 'hasta' (lado de rechazo), donde |H| cae bajo 'nivel'
    m = (f >= min(desde, hasta)) & (f <= max(desde, hasta))
    ff, mm = f[m], mag[m]
    if desde > hasta:
        ff, mm = ff[::-1], mm[::-1]
    i = int(np.argmax(mm < nivel))
    if i == 0 or mm[i] >= nivel:
        return np.nan
    return np.interp(nivel, [mm[i], mm[i - 1]], [ff[i], ff[i - 1]])

def transitorio(ba, fs, T=30):
    # tiempo hasta que la respuesta al impulso del IIR cae (y se queda) bajo el 0.1 % de su máximo
    x = np.zeros(int(T * fs)); x[0] = 1
    h = np.abs(filtrar_iir(ba, x))
    return np.nonzero(h > 1e-3 * h.max())[0][-1] / fs

def mediciones(dis, fs):
    filas = {}
    for nombre, d in dis.items():
        C = TABLA_VENTANAS.loc[d['ventana'], 'C']
        wF, hF, _, _ = respuesta(d['b'], 'fir', fs, 2**17)
        wI, hI, _, gdI = respuesta(d['ba'], 'iir', fs, 2**17)
        mF, mI = 20 * np.log10(np.abs(hF) + 1e-15), 20 * np.log10(np.abs(hI) + 1e-15)
        fps, fsts = np.atleast_1d(d['fp']), np.atleast_1d(d['fst'])
        paso = bandas_paso_rechazo(wI, d)[0]
        for k, (fp_, fst_) in enumerate(zip(fps, fsts)):
            ext = 0.5 * abs(fp_ - fst_)
            desde = np.clip(fp_ + np.sign(fp_ - fst_) * ext, 0, fs / 2)
            hasta = np.clip(fst_ - np.sign(fp_ - fst_) * ext, 0, fs / 2)
            filas[(nombre, f'borde {fp_}→{fst_} Hz')] = {
                'FIR f(−3 dB)': cruce(wF, mF, -3, desde, hasta),
                'FIR f(−6 dB)': cruce(wF, mF, -6, desde, hasta),
                'IIR f(−3 dB)': cruce(wI, mI, -3, desde, hasta),
                'FIR Δf medido (−0.1→−50 dB)': abs(cruce(wF, mF, -AS, desde, hasta) - cruce(wF, mF, -RP, desde, hasta)),
                'FIR Δf teoría C·fs/(2(Lh−1))': C * fs / (2 * (len(d['b']) - 1)),
                'IIR Δf medido (−0.1→−50 dB)': abs(cruce(wI, mI, -AS, desde, hasta) - cruce(wI, mI, -RP, desde, hasta)),
                'FIR τg [ms]': 1000 * (len(d['b']) - 1) / 2 / fs,
                'IIR τg mín/máx en paso [ms]': f"{1000 * np.nanmin(gdI[paso]) / fs:.1f} / {1000 * np.nanmax(gdI[paso]) / fs:.1f}",
                'IIR transitorio [s]': transitorio(d['ba'], fs)}
    return pd.DataFrame(filas).T.round(2)

print("ECG"); med_ecg = mediciones(dis_ecg, fs_e); display(med_ecg)
print("PCG"); med_pcg = mediciones(dis_pcg, fs_p); display(med_pcg)

# Recorte para las métricas: el mayor entre la mitad del FIR más largo y el transitorio del IIR más largo
RECORTE_E = np.ceil(max(max((len(d['b']) - 1) / 2 / fs_e for d in dis_ecg.values()), med_ecg['IIR transitorio [s]'].max()))
RECORTE_P = np.ceil(max(max((len(d['b']) - 1) / 2 / fs_p for d in dis_pcg.values()), med_pcg['IIR transitorio [s]'].max()))
print(f"Recorte para métricas: ECG {RECORTE_E:.0f} s, PCG {RECORTE_P:.0f} s por extremo")

#### Comparación entre tipos de filtro:
Al analizar los valores obtenidos anteriormente, vemos que:

- **Frecuencia de corte:**

En el FIR, el punto de −6 dB cae exactamente en la $f_c$ de diseño, lo que confirma que $\Omega_c$ es el centro de la transición. El −3 dB del FIR queda ligeramente hacia adentro (37.6 Hz en el pasa-bajo, 0.549 Hz en el pasa-alto). En el elíptico el −3 dB está más pegado a $f_p$: 33.0 Hz en el pasa-bajo, 0.635 Hz en el pasa-alto, 7.7/12.5 Hz en el QRS, 366 Hz y 31.7 Hz en los del PCG. Como los dos cumplen las especificaciones planteadas, "40 Hz" no significa lo mismo en uno y en otro: lo que se garantiza es $f_p$ y $f_{st}$. En el QRS la diferencia es grande: el FIR deja pasar 5.7–14.3 Hz y el elíptico solo 7.7–12.5 Hz (ver la nota de polos y ceros).

- **Ripple y atenuación:**

El FIR tiene ripple chico en la banda de paso (0.03–0.04 dB) y lóbulos en la de rechazo que arrancan en ~−53 dB junto al borde y bajan lentamente al alejarse. El elíptico es **equiripple en las dos bandas**: ondula 0.1 dB en la de paso y sus lóbulos de rechazo se quedan en −50 dB en toda la banda, sin seguir bajando. Por eso, lejos del borde, el FIR atenúa más (en el PCG, la banda de 450–1000 Hz baja 75 dB con el FIR y 53 dB con el elíptico).

- **Ancho de transición:**

El medido en el FIR es un 5–8 % menor que el de la teoría, porque la constante $C$ de la tabla corresponde a $\alpha_s=54.5$ dB y aquí se mide hasta −50 dB. El elíptico logra un ancho igual o menor con un orden muchísimo más bajo: 19.5 Hz contra 18.8 Hz en el pasa-bajo (orden 5 contra 84), 0.28 Hz contra 0.37 Hz en el pasa-alto (orden 5 contra 4150), 90 Hz contra 91 Hz en el pasa-bajo del PCG (orden 6 contra 68). En los pasa-banda del FIR las dos transiciones salen iguales, así que el borde de 40 Hz del ECG queda con una transición de **0.37 Hz**, mucho más angosta de lo pedido (sobrediseño impuesto por el borde de 0.5 Hz), y lo mismo pasa en el borde de 400 Hz del PCG (18.7 Hz). El IIR en cambio le da a cada borde aproximadamente lo que necesita (11.5 y 78 Hz).

- **Fase:**

En el FIR el retardo de grupo es una recta horizontal: 84 ms en el pasa-bajo, **4.15 s** en el pasa-alto y el pasa-banda del ECG, 282 ms en el QRS, 830 ms en el notch, y 17 y 83 ms en el PCG. En el elíptico el retardo es menor en promedio pero **varía con la frecuencia** y crece hacia los bordes: 16→46 ms dentro de la banda de paso del pasa-bajo, hasta 1.9 s cerca de 0.7 Hz en el pasa-alto, 178→462 ms en el QRS, y de 0.1 ms lejos de 50 Hz a 240 ms en los bordes del notch. Componentes de distinta frecuencia (P y T de ~1–10 Hz, QRS de ~10–30 Hz) se retrasan distinto, y eso deforma la onda.

- **Transitorios:**

En el IIR: 0.28 s en el pasa-bajo, 1.3 s en el pasa-alto de 0.5 Hz, **5.9 s** en el pasa-banda del ECG (lo dominan los polos de la etapa pasa-alto, a 0.001 del círculo unidad), 2.7 s en el QRS, 1 s en el notch y menos de 0.4 s en el PCG. En el FIR los primeros y últimos $(L_h-1)/2$ valores son convoluciones parciales: hasta 4.15 s en el ECG. Por eso las métricas recortan **6 s** por extremo en el ECG y **1 s** en el PCG.

- **Costo**:

En multiplicaciones por muestra (FIR: $L_h$, o $(L_h+1)/2$ aprovechando la simetría; elíptico en forma directa: $2N+1$ por etapa): pasa-alto de 0.5 Hz, 4151 contra 11; notch, 831 contra 17; pasa-bajo de 40 Hz, 85 (43) contra 11; pasa-bajo del PCG, 69 (35) contra 13. Con el elíptico, el IIR es más barato en todos los casos. La ventaja es enorme cuando la transición es angosta respecto de $f_s$ (pasa-alto de 0.5 Hz, notch) y moderada cuando es ancha (pasa-bajos), porque $L_h\propto f_s/\Delta f$ depende del ancho **absoluto** de la transición, mientras que el orden del IIR depende del cociente $f_{st}/f_p$.

Luego vemos que todos los tipos de filtro se pueden implementar tanto con FIR como con IIR, pero lo que cambia es el costo de cumplir la misma especificación.
Luego para ilustrar este costo, se calcula el largo del FIR, el orden del IIR para las tres familias (Butterworth, Chebyshev y elíptico), y las multiplicaciones por muestra. En el FIR son $L_h$, o $(L_h+1)/2$ si se aprovecha la simetría sumando primero $x[n-k]+x[n-L_h+1+k]$. En el IIR en forma directa son $2N+1$ por etapa: $N+1$ coeficientes $b_k$ y $N$ coeficientes $a_k$ ($a_0=1$).

In [ ]:
def costos(espec, dis, fs):
    filas = {}
    for nombre, e in espec.items():
        d = dis[nombre]
        _, Df = bandas(e)
        nt_k = signal.kaiserord(AS, Df / (fs / 2))[0]; nt_k += 1 - nt_k % 2      # Kaiser: δp = δs = 10^(−As/20)
        ba_b, N_b = disenar_iir(e, fs, familia='butter')
        ba_c, N_c = disenar_iir(e, fs, familia='cheby1')
        filas[nombre] = {'Δf [Hz]': Df,
                         'FIR Hamming: Lh': len(d['b']), 'FIR Kaiser: Lh': nt_k,
                         'IIR Butterworth: orden': N_b, 'Butterworth (b, a) estable y cumple': estable(ba_b) and cumple(ba_b, e, fs),
                         'IIR Chebyshev I: orden': N_c, 'Chebyshev I (b, a) estable y cumple': estable(ba_c) and cumple(ba_c, e, fs),
                         'IIR elíptico: orden': d['N'],
                         'mult/muestra FIR Hamming': len(d['b']),
                         'mult/muestra FIR (aprovechando simetría)': (len(d['b']) + 1) // 2,
                         'mult/muestra Butterworth': mult_iir(ba_b),
                         'mult/muestra elíptico': mult_iir(d['ba'])}
    return pd.DataFrame(filas).T

print(f"Misma plantilla para todos (rp = {RP} dB, αs = {AS} dB)")
print(f"ECG, fs = {fs_e} Hz"); display(costos(espec_ecg, dis_ecg, fs_e))
print(f"PCG, fs = {fs_p} Hz"); display(costos(espec_pcg, dis_pcg, fs_p))

e_hp = espec_ecg['Pasa-alto 0.5 Hz']
ba_8k, N_8k = disenar_iir(e_hp, fs)
print(f"\nPasa-alto 0.5 Hz SIN submuestrear (fs = {fs} Hz): FIR Hamming Lh = {largo_fir(bandas(e_hp)[1], fs, TABLA_VENTANAS.loc['hamming', 'C'])} "
      f"| elíptico orden {N_8k}, |polo| máx exacto = {np.max(np.abs(polos_exactos(e_hp, fs, familia='ellip'))):.7f}, "
      f"estable en (b, a): {estable(ba_8k, n=2**20)}")

**Comentario de la tabla de costos:** en forma directa, el Butterworth no es utilizable en 4 de los 5 filtros del ECG, y el Chebyshev I falla en el pasa-alto y el pasa-banda: cuanto más alto el orden y más cerca de $z=1$ están los polos, peor. El elíptico cumple con órdenes 5 a 12 y es el más barato en todos los casos. La estimación de Kaiser da FIR un 10–12 % más cortos que Hamming, porque puede apuntar justo a 50 dB en lugar de 54.5 dB, pero sigue en el mismo orden de magnitud.

**Sin submuestrear**, el pasa-alto de 0.5 Hz a 8 kHz necesitaría un FIR de **66 401 coeficientes**, y el elíptico, aunque sigue siendo de orden 5 (el orden no depende de $f_s$), tiene polos en $|p|=0.99994$ y en forma $(b,a)$ ya **no es estable**. Esto justifica el submuestreo previo.

## Aplicación de los filtros

Luego para cada filtro, se aplica a las señales correspondientes y se mide la densidad espectral de potencia antes y después del filtrado sobre la banda relevante. 
Las métricas se calculan recortando los transitorios medidos arriba (6 s por extremo en el ECG y 1 s en el PCG):

- **ΔP [dB]:** cambio de potencia en cada banda respecto de la señal contaminada (negativo = atenuó). Se espera una caída grande en las bandas de ruido y ~0 dB en la banda útil.
- **SNR vs original [dB]** y **correlación:** qué tan parecida es la salida a la señal original. Es decir, se mide la relación señal ruido utilizando la potencia de las señales. Ojo: la salida del FIR tiene el retardo compensado (`mode='same'`, procesamiento offline), mientras que el IIR es causal, así que en esta métrica el IIR paga también su retardo.
- **Tabla lineal:** como $H\{x+r\}=H\{x\}+H\{r\}$, se filtra cada componente por separado. Así se mide exactamente la **atenuación de cada contaminación** y la **SNR de distorsión** de la señal ($\mathrm{var}(x)/\mathrm{var}(H\{x\}-x)$), sin que una se mezcle con la otra.

In [ ]:
def filtrar_todos(x, dis):
    res = {}
    for nombre, d in dis.items():
        res[nombre] = {'FIR': signal.oaconvolve(x, d['b'], mode='same'),
                       'IIR': filtrar_iir(d['ba'], x)}           # lfilter(b, a, x)
    return res

res_ecg = filtrar_todos(ecg_cont, dis_ecg)
res_pcg = filtrar_todos(pcg_cont, dis_pcg)

comp_ecg = {'señal ECG': ecg_d, 'deriva 0.3 Hz': deriva, 'red 50 Hz': red_50}
comp_pcg = {'señal PCG': pcg_d, 'ruido blanco': blanco, 'interferencia 10 Hz': baja}
lin_ecg = tabla_lineal(comp_ecg, filtros_de(dis_ecg), fs_e, RECORTE_E)
lin_pcg = tabla_lineal(comp_pcg, filtros_de(dis_pcg), fs_p, RECORTE_P)
print("ECG: atenuación de cada contaminación y SNR de distorsión del ECG (filtrando cada componente por separado)")
display(lin_ecg)
print("PCG: ídem")
display(lin_pcg)

### ECG

#### Pasa banda 0.5-40Hz 
Elimina a la vez la deriva (por debajo de 0.5 Hz) y la red de 50 Hz (por encima de 40 Hz). Se muestra un zoom de unos pocos latidos en el centro del registro, donde se ve la forma del QRS. 

In [ ]:
nombre = 'Pasa-banda 0.5-40 Hz'
y = res_ecg[nombre]
curvas = [('contaminada', ecg_cont, 'C0'), ('FIR', y['FIR'], 'darkorange'), ('IIR elíptico', y['IIR'], 'C3')]

graficar_tiempo(t_e, curvas, f'ECG - {nombre}: zoom de 4 s en el centro del registro', zoom=ZOOM, ref=ecg_d)
plt.show()

In [ ]:
graficar_espectro(fs_e, [('contaminada', ecg_cont, 'C0'), ('FIR', y['FIR'], 'darkorange'), ('IIR elíptico', y['IIR'], 'C3')],
               f'ECG - {nombre}: espectro antes y después', fmax=fs_e / 2, fmin=0.1, log=True)

bandas_ecg = {'deriva 0-0.5 Hz': (0, 0.5), 'útil 0.7-30 Hz': (0.7, 30), 'red 48-52 Hz': (48, 52), 'alta 60-250 Hz': (60, 250)}
plt.show()
display(tabla_metricas(ecg_d, ecg_cont, y, fs_e, bandas_ecg, RECORTE_E))

# Retardo del IIR en la banda del QRS y desplazamiento del pico R (correlación cruzada con la original)
d = dis_ecg[nombre]
w_, h_, _, gd_ = respuesta(d['ba'], 'iir', fs_e, 2**15)
r = int(RECORTE_E * fs_e); sl = slice(r, -r)
lags = np.arange(-100, 101)
xc = [np.dot(ecg_d[sl], np.roll(y['IIR'], -k)[sl]) for k in lags]
print(f"IIR: τg(10 Hz) = {1000 * np.interp(10, w_, gd_) / fs_e:.1f} ms, τg(1 Hz) = {1000 * np.interp(1, w_, gd_) / fs_e:.1f} ms | "
      f"retardo que maximiza la correlación con el original: {1000 * lags[np.argmax(xc)] / fs_e:.0f} ms")

Vemos que se atenúan las frecuencias adecuadamente: la deriva baja 55.5 dB con el FIR y 51.7 dB con el IIR, es decir que de $0.5A$ queda del orden de $0.0013A$. La red baja 81.7 y 55 dB. En el zoom, la oscilación lenta de la línea de base y la de 50 Hz desaparecen en las dos salidas.

- **Retardo de grupo:**

Como el retardo del FIR es una constante conocida, $(L_h-1)/2$, se compensa, y la salida queda alineada gracias a la fase lineal. Esto equivale a procesar *offline*: en tiempo real, la salida del FIR llegaría con 4.15 s de atraso.

- **Forma y amplitud del QRS:**

El FIR (retardo compensado) queda **alineado** con el original y se superpone casi perfectamente en el zoom (SNR 16.6 dB, correlación 0.99). Lo único que pierde es el contenido entre ~30 y 40 Hz y lo que está por encima, que en este ECG es el 1.94 % de la potencia (impreso al principio), y se nota en un pico R un poco más bajo. Ese 1.94 % pone además un techo a la SNR de cualquier filtro que corte en 40 Hz: $10\log_{10}(1/0.0194)\approx 17$ dB.

El IIR, por el otro lado, está **retrasado y deformado** (SNR −2.0 dB, correlación 0.18): su retardo de grupo es de ~22 ms en 10 Hz (QRS) pero de ~0.44 s en 1 Hz. Como las ondas lentas (P, T, línea de base) se retrasan mucho más que el QRS, la relación temporal entre ondas se deforma. En el zoom se ve que no es solo un corrimiento: aparece una oscilación después del QRS y la onda T cambia de forma.

- **Transitorio inicial:**

En el IIR dura ~**5.9 s**, mucho más que en el resto, porque los polos de la etapa pasa-alto están a 0.001 del círculo unidad y su contribución decae como $|p|^n$, es decir, muy lentamente. El FIR solo tiene efecto de borde en los primeros y últimos 4.15 s (convolución parcial).

Finalmente, vemos que el FIR conserva mejor la morfología: el IIR queda último por la fase, no porque elimine peor las interferencias. El retardo que maximiza la correlación con el original es de 18 ms: corriendo la salida esa cantidad, el QRS se superpone, pero la P y la T siguen desplazadas, porque el retardo no es el mismo para todas las frecuencias. El IIR es la opción para tiempo real con poco costo (24 multiplicaciones por muestra contra 4151) si se acepta esa distorsión de fase. Si el procesamiento es offline, filtrar hacia adelante y hacia atrás (`filtfilt`) anularía la fase del IIR.

#### Pasa-bajo, pasa-alto, QRS y notch
Se repite el análisis con los demás filtros para ver qué hace cada uno por separado (FIR comparado con IIR). En el notch se ve que la red desaparece **sin recortar** la banda de 40 a 250 Hz del ECG, a diferencia del pasa-banda.

In [ ]:
for nombre in ['Pasa-bajo 40 Hz', 'Pasa-alto 0.5 Hz', 'Pasa-banda QRS 5-15 Hz', 'Notch 50 Hz']:
    y = res_ecg[nombre]
    curvas = [('contaminada', ecg_cont, 'C0'), ('FIR', y['FIR'], 'darkorange'), ('IIR elíptico', y['IIR'], 'C3')]
    graficar_tiempo(t_e, curvas, f'ECG - {nombre}: zoom de 4 s', zoom=ZOOM, ref=ecg_d)
    d = dis_ecg[nombre]
    graficar_espectro(fs_e, curvas, f'ECG - {nombre}: espectro', fmin=d['xlim'][0], fmax=d['xlim'][1], log=d['log'])
    plt.show()
    print(nombre); display(tabla_metricas(ecg_d, ecg_cont, y, fs_e, bandas_ecg, RECORTE_E))

**Comparación entre filtros del ECG:**

- **Cada filtro resuelve una sola contaminación.** El pasa-bajo elimina la red (−53 dB con los dos) y deja la deriva intacta (ΔP de 0 dB en 0–0.5 Hz); el pasa-alto elimina la deriva y deja la red; el notch elimina la red y deja la deriva. Por eso, con un solo filtro, solo el pasa-banda (o el QRS) resuelve las dos.
- **Pasa-bajo:** por encima de 60 Hz el FIR baja 62 dB y el elíptico 52 dB (se queda en su piso de −50 dB). En el espectro se ve que el elíptico corta antes (−3 dB en 33 Hz contra 37.6 Hz del FIR).
- **Pasa-banda QRS:** elimina la deriva y la red, pero también la mayor parte de las ondas P y T y la parte del QRS por encima de 15 Hz (ΔP de −6.2 dB con el FIR y −8.9 dB con el IIR en la banda útil 0.7–30 Hz). A cambio, los picos R quedan marcados y aislados, listos para detectarlos (Pan y Tompkins). El IIR deja una salida más chica y más oscilante que el FIR porque su banda de paso real es mucho más angosta (7.7–12.5 Hz contra 5.7–14.3 Hz) y su retardo varía entre 178 y 462 ms dentro de la banda, lo que desplaza los picos (correlación −0.05). Para detectar picos R conviene el FIR, con retardo constante de 282 ms que se puede restar.
- **Notch:** conserva el contenido del ECG entre 40 y 250 Hz (ΔP de 0 dB en 60–250 Hz) que el pasa-bajo y el pasa-banda recortan, y en el espectro se ve la ranura angosta en 50 Hz. El ΔP de la banda de 48–52 Hz es solo de −36 (FIR) y −43 dB (IIR) porque esa banda es más ancha que la ranura y el contenido propio del ECG en sus bordes queda; filtrando la red sola (tabla lineal), la atenuación es de 61 y 50 dB. **No resuelve la deriva.**

### PCG 

#### Pasa-banda 25–400 Hz

In [ ]:
nombre = 'Pasa-banda 25-400 Hz'
y = res_pcg[nombre]
curvas = [('contaminado', pcg_cont, 'C0'), ('FIR', y['FIR'], 'darkorange'), ('IIR elíptico', y['IIR'], 'C3')]

graficar_tiempo(t_p, curvas, f'PCG - {nombre}: zoom de 4 s', zoom=ZOOM, ref=pcg_d, ylim=YLIM_PCG)

graficar_espectro(fs_p, [(l, x, c) for l, x, c in curvas], f'PCG - {nombre}: espectro antes y después',
               fmax=fs_p / 2, fmin=1, log=True)
plt.show()

bandas_pcg = {'baja 0-15 Hz': (0, 15), 'útil 35-350 Hz': (35, 350), 'alta 450-1000 Hz': (450, 1000)}
display(tabla_metricas(pcg_d, pcg_cont, y, fs_p, bandas_pcg, RECORTE_P))

A continuación se grafica el espectograma de la señal. Es decir, la energía de la señal en función del tiempo (eje x) y la frecuencia (eje y), en dB. Se calcula con la FFT de ventanas cortas (256 muestras, con solapamiento) que se van deslizando sobre la señal.

El espectrograma permite ver cómo se distribuye la energía de S1 y S2 en el tiempo y la frecuencia, y que el filtro remueve las bandas sin información (la franja de 10 Hz abajo y el ruido arriba) sin tocar los sonidos cardíacos. Debajo se imprime cuánta potencia del PCG original en la banda de máxima energía de S1 (25–45 Hz, según el TP2) conserva el pasa-banda.

In [ ]:
def graficar_espectrogramas(fs, curvas, titulo, fmax, t_lim=None):
    fig, axs = plt.subplots(len(curvas), 1, figsize=(12, 3 * len(curvas)), sharex=True)
    axs = np.atleast_1d(axs)
    S_all = []
    for label, x in curvas:
        if t_lim:
            x = x[int(t_lim[0] * fs):int(t_lim[1] * fs)]
            t0 = t_lim[0]
        else:
            t0 = 0
        f, t, S = signal.spectrogram(x, fs, nperseg=256, noverlap=192)
        S_all.append((label, f, t + t0, 10 * np.log10(S + 1e-20)))
    vmax = max(s.max() for _, _, _, s in S_all); vmin = vmax - 80
    for ax, (label, f, t, S) in zip(axs, S_all):
        im = ax.pcolormesh(t, f, S, shading='auto', vmin=vmin, vmax=vmax, cmap='magma')
        ax.set_ylim(0, fmax); ax.set_ylabel('f [Hz]'); ax.set_title(label, fontsize=10)
        fig.colorbar(im, ax=ax, label='dB')
    axs[0].set_title(f'{titulo}\n{S_all[0][0]}', fontsize=10)
    axs[-1].set_xlabel('t (s)')
    plt.tight_layout()
graficar_espectrogramas(fs_p, [('PCG original', pcg_d), ('PCG contaminado', pcg_cont),
                               ('PCG filtrado (IIR elíptico, 25-400 Hz)', y['IIR'])],
                        'PCG: espectrograma', fmax=1000, t_lim=(ZOOM[0] - 2, ZOOM[1] + 2))

# Cuánto de la banda de máxima energía de S1 (25-45 Hz) conserva el pasa-banda (filtrando solo el PCG original)
r = int(RECORTE_P * fs_p); sl = slice(r, -r)
for metodo, fn in filtros_de(dis_pcg, [nombre]).items():
    pcg_f = fn(pcg_d)
    print(f"{metodo}: conserva el {100 * potencia_banda(pcg_f[sl], fs_p, 25, 45) / potencia_banda(pcg_d[sl], fs_p, 25, 45):.1f} % "
          f"de la potencia del PCG en 25-45 Hz y el {100 * np.var(pcg_f[sl]) / np.var(pcg_d[sl]):.1f} % de la potencia total")

Vemos que **S1 y S2 siguen bien definidos en el tiempo**: las "manchas" verticales de cada sonido aparecen en los mismos instantes antes y después de filtrar, y en la señal filtrada sobresalen claramente del ruido que queda dentro de la banda (el ruido agregado está 9.8 dB por debajo del PCG, ver la línea impresa más abajo). Desaparecen la franja horizontal de 10 Hz y el piso de ruido por encima de ~450 Hz. El retardo del IIR en la banda del PCG va de 2.5 a 80 ms y llega a ~100 ms cerca de 33 Hz, justo en la banda de S1: no se ve a la escala del espectrograma, pero en el gráfico temporal se nota que los sonidos del IIR aparecen algo corridos y con más oscilación que los del FIR. Si se quisiera medir la relación temporal con el ECG, ese retardo no es despreciable.

Sin embargo, el pasa-alto recorta parte de S1, dado que parte de la banda de 25–45 Hz cae en la transición. Lo impreso arriba cuantifica cuánta potencia de esa banda se conserva: 83 % con el FIR y 51 % con el IIR, que corta más porque su −3 dB está en 33.4 Hz contra 27.4 Hz del FIR. Del PCG total se conserva solo el 33 % (FIR) y el 24 % (IIR), porque el 63 % de su potencia está por debajo de 25 Hz: los sonidos filtrados se ven más chicos que el original en el gráfico temporal. Es el costo de poner la interferencia de 10 Hz en la banda de rechazo. Si se quisiera conservar más S1, se podría usar $f_p=25$, $f_{st}=12$ Hz (sigue rechazando 10 Hz), a cambio de un FIR de $1+3.32\cdot 2000/13\approx 513$ coeficientes en lugar de 333.

Por el otro lado, en la banda útil 35–350 Hz el ΔP queda en ~0 dB (el ripple es de 0.1 dB como máximo). La banda baja (0–15 Hz) cae 65 dB (FIR) y 59 dB (IIR); no llega a la atenuación de la senoidal de 10 Hz porque en esa banda también está el contenido grave del PCG, que cae en la transición del filtro. La alta (450–1000 Hz) cae 75 dB (FIR) y 53 dB (IIR): el elíptico se queda en los ~50 dB de la plantilla, mientras que los lóbulos del FIR siguen bajando lejos del borde.

#### Pasa-bajo y pasa-alto por separado

In [ ]:
for nombre in ['Pasa-bajo 400 Hz', 'Pasa-alto 25 Hz']:
    y = res_pcg[nombre]
    graficar_tiempo(t_p, [('contaminado', pcg_cont, 'C0'), ('FIR', y['FIR'], 'darkorange'), ('IIR elíptico', y['IIR'], 'C3')],
                    f'PCG - {nombre}: zoom de 4 s', zoom=ZOOM, ref=pcg_d, ylim=YLIM_PCG)
    plt.show()
    print(nombre); display(tabla_metricas(pcg_d, pcg_cont, y, fs_p, bandas_pcg, RECORTE_P))

# Potencia de cada contaminación respecto de la del PCG (antes de filtrar)
P_pcg = np.var(pcg_d)
print(f"Potencia relativa al PCG: interferencia 10 Hz {10*np.log10(np.var(baja)/P_pcg):.1f} dB | ruido blanco {10*np.log10(np.var(blanco)/P_pcg):.1f} dB")

A partir de lo anterior vemos que el **pasa-alto** aporta más a la mejora de SNR: La interferencia de 10 Hz tiene una potencia de $0.045A^2$, 12.6 dB más que todo el ruido blanco ($0.0025A^2$; la diferencia también se ve en la línea impresa, que da ambas potencias relativas al PCG). El pasa-alto la elimina (−78 dB con el FIR, −52 dB con el IIR), mientras que el pasa-bajo solo puede sacar la parte del ruido que está por encima de 400 Hz: −4.1 dB, es decir, el 61 % de su potencia. Por eso, en la tabla de métricas, el pasa-bajo solo casi no mejora la SNR respecto de la señal contaminada (de −2.9 a −2.8 dB con el FIR), y el pasa-alto la mejora claramente (a +1.6 dB), aunque también pierde la parte del PCG que está por debajo de 35 Hz.

Como las dos contaminaciones están en bandas distintas y el pasa-banda las ataca a la vez, el pasa-banda solo es mejor que cada uno por separado: elimina la interferencia de 10 Hz como el pasa-alto **y además** el ~60 % del ruido como el pasa-bajo. Su SNR es la mayor de los tres (+2.0 dB con el FIR). Aun así queda acotada, y no principalmente por el ruido: el que queda dentro de 25–400 Hz está ~14 dB por debajo del PCG (9.8 dB de partida más los 4.3 dB que saca el filtro). Lo que más pesa es la parte del PCG que se pierde: el 63 % de su potencia está por debajo de 25 Hz, en la misma zona que la interferencia de 10 Hz, y el pasa-alto se la lleva junto con ella. Eso cuenta como error contra el original.

Como límitación, sin embargo, vemos que el ruido blanco ocupa todas las frecuencias (también las de 25–400 Hz donde está el PCG), y ningún filtro de frecuencia puede separarlos dentro de esa banda. Lo mismo pasa con las ondas P, QRS y T del ECG, que se solapan en frecuencia: por eso el filtrado debe combinarse con información temporal (detección de picos R, ventanas, promediado de latidos). Esto también fue explorado en el trabajo anterior a partir del sumador y promediador.

## Conclusiones 

1. **FIR vs IIR para cada señal.** Todos los tipos de filtro, notch incluido, se pueden hacer con ambos. Con la misma plantilla ($r_p=0.1$ dB, $\alpha_s=50$ dB) lo que cambia es el costo, y depende de qué tan angosta es la transición respecto de $f_s$, porque $L_h\propto f_s/\Delta f$ y el orden del IIR depende de $f_{st}/f_p$.
   - **ECG:** el pasa-alto de 0.5 Hz y el notch son casos claros para el **IIR**: 4151 y 831 coeficientes de FIR (4.15 s y 0.83 s de retardo) contra órdenes 5 y 8 del elíptico (11 y 17 multiplicaciones por muestra), o incluso un `iirnotch` de orden 2. El costo es la fase no lineal.
   - **PCG:** las transiciones son relativamente anchas ($\Delta f/f_s$ = 0.01–0.05) y el **FIR**, aunque sigue siendo más caro que el elíptico (167 multiplicaciones por muestra aprovechando la simetría contra 25 en el pasa-banda, y 35 contra 13 en el pasa-bajo), tiene un costo absoluto bajo. Con fase lineal y solo 83 ms de retardo, es la mejor opción si se quiere conservar la relación temporal entre S1, S2 y el ECG (lo cual es especialmente relevante dado que se trata de señales médicas)
   - **Familia de IIR:** con la plantilla común, el Butterworth necesita órdenes 10–42 y en forma $B(z)/A(z)$ se vuelve inestable o deja de cumplir la plantilla en 5 de los 8 filtros, solo por el redondeo de los coeficientes. Por eso se usa el **elíptico**, que cumple la misma plantilla con órdenes 5–12 y es estable en todos los casos (el pasa-banda del ECG, en dos etapas $(b,a)$).
2. **Fase.** El FIR retrasa todo $(L_h-1)/2$ muestras: se compensa y la salida queda alineada (`mode='same'`). El IIR retrasa cada frecuencia distinto: ~22 ms el QRS y ~0.44 s la banda de 1 Hz en el pasa-banda del ECG. Desplaza y deforma las ondas, y por eso su SNR contra el original es la peor, aunque atenúe las interferencias igual que el FIR. La fase **importa** cuando se miden tiempos o formas: detección de picos R, intervalo RR, segmento S-T, y la relación temporal entre el ECG y el PCG. Sin embargo, no es tan importante en la detección de anomalías como soplos, o cuando solo interesa la energía por banda.
3. **Trade-offs.**
   - **Orden vs selectividad:** en el FIR, el ancho de transición baja como $1/L_h$ pero los lóbulos no bajan. En el IIR, más orden da más selectividad, pero también más retardo de grupo y, en forma $B(z)/A(z)$, más sensibilidad al redondeo de los coeficientes: el Butterworth de órdenes 10 a 42 falla en 5 de los 8 filtros.
   - **Ancho de transición vs atenuación (ventanas):** según la tabla de la teoría, de la rectangular a Blackman la atenuación pasa de 20.9 a 75.3 dB y $C$ de 1.84 a 11.13. Con la plantilla de 50 dB, la ventana mínima es Hamming, y las mediciones lo confirman (51–53 dB).
   - **Selectividad vs rizado y fase (familias IIR):** con la misma plantilla, el elíptico necesita orden 5 contra 15 del Butterworth en el pasa-bajo de 40 Hz, a cambio de ripple en las dos bandas (su rechazo no pasa de −50 dB) y de picos de retardo de grupo junto a los bordes (~55 ms en 32 Hz).
   - **Recortar banda vs quitar solo la interferencia:** el pasa-banda resuelve la deriva y la red a la vez pero se lleva todo lo que está por encima de 40 Hz; el notch conserva 40–250 Hz y quita solo la red, pero no resuelve la deriva.
4. **Limitaciones.** Ningún filtro de frecuencia separa señal e interferencia cuando ocupan la misma banda: la interferencia de 10 Hz cae donde está el 63 % de la potencia de este PCG, y el ruido que cae dentro de la banda de la señal no se puede filtrar: con el pasa-banda del PCG queda el 37 % del ruido (casi todo dentro de 25–400 Hz), en el límite del filtro ideal. Las ondas del ECG (P, QRS, T) y los sonidos del PCG (S1, S2) se solapan en frecuencia, así que ningún filtro las separa: hace falta información temporal. Además, los IIR con polos muy cerca de $z=1$ tienen transitorios de varios segundos.
5. **Sobre la comparación.** El FIR se aplicó con su retardo compensado (offline) y el IIR de forma causal (como en tiempo real), así que en morfología y SNR la comparación favorece al FIR. En tiempo real, el FIR del pasa-banda del ECG llegaría con 4.15 s de atraso.